<a href="https://colab.research.google.com/github/dee0742/ML-FlyRank-Task/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip -q install duckdb

In [15]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [16]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [17]:
rel = "hf://datasets/FlyRank/internship-warehouse"

march_path = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"

In [18]:
import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dee0742/ML-FlyRank-Task/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [19]:

march_path = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"

con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM read_parquet('{march_path}')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬─────────────────┬─────────────────┐
│ row_count │ min_report_date │ max_report_date │
│   int64   │      date       │      date       │
├───────────┼─────────────────┼─────────────────┤
│   9841378 │ 2026-03-01      │ 2026-03-31      │
└───────────┴─────────────────┴─────────────────┘


One row represents one pseudonymized content item for one pseudonymized client on one reporting date.

The main table is `fact_content_daily_performance`, whose documented grain is:

`report_date × client_id × content_id`

For development, I will use the March 2026 partition. I will keep the final June 2026 month as a sealed test window rather than using it to develop the label.

The prediction/ranking decision must use information available before the outcome window.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [20]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{march_path}')
LIMIT 1
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │


### Features

I will use up to five fields that are available before the prediction/ranking decision.

The final five features will be selected after checking the warehouse schema and their time windows. I will not use any field derived from the future outcome.

### Label / proxy

The label will represent a future observed outcome. It must be measured in a later time window than the features.

I will not use `is_declining_label` as a model feature because it is a derived label.

### Context

- `client_id` — used for grouping, joining, and client-level train/test splitting.
- `content_id` — used for identifying and joining content.
- `report_date` — identifies the observation date.
- `content_type` — used for grouping and checking missingness.

### Excluded

- `trend_pct` — excluded because it is used to derive the declining outcome.
- `trend_direction` — excluded because it is used to derive `is_declining_label`.
- `is_declining_label` — excluded from features because it is a derived label.
- `client_id` and `content_id` — excluded from model features because they are pseudonymous identifiers.
- Future-window measurements — excluded because they would not be available at decision time.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Grain check

The documented grain is one row per `report_date × client_id × content_id`.

The following query looks for duplicate combinations. An empty result supports the claimed grain.

In [22]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{march_path}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────────┐
│ report_date │ client_hash_id │ content_hash_id │ row_count │
│    date     │    varchar     │     varchar     │   int64   │
├─────────────┴────────────────┴─────────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

### March 2026 slice

This checks how many rows are in the development slice and the observed reporting-date range.

In [23]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM read_parquet('{march_path}')
""")

┌───────────┬─────────────────┬─────────────────┐
│ row_count │ min_report_date │ max_report_date │
│   int64   │      date       │      date       │
├───────────┼─────────────────┼─────────────────┤
│   9841378 │ 2026-03-01      │ 2026-03-31      │
└───────────┴─────────────────┴─────────────────┘

### Availability and missingness

GA4 fields are only usable when `ga4_data_available IS TRUE`. A zero in a GA4 field is not automatically zero engagement.

I will also check missingness for the fields used in the contract rather than blindly replacing missing values with zero.

In [25]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS NOT TRUE
    ) AS ga4_unavailable_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,

    AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END)
        AS gsc_clicks_missing_rate,

    AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END)
        AS gsc_impressions_missing_rate,

    AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END)
        AS gsc_avg_position_missing_rate

FROM read_parquet('{march_path}')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬──────────────────────┬────────────────────┬─────────────────────────┬──────────────────────────────┬───────────────────────────────┐
│ total_rows │ ga4_available_rows │ ga4_unavailable_rows │ gsc_available_rows │ gsc_clicks_missing_rate │ gsc_impressions_missing_rate │ gsc_avg_position_missing_rate │
│   int64    │       int64        │        int64         │       int64        │         double          │            double            │            double             │
├────────────┼────────────────────┼──────────────────────┼────────────────────┼─────────────────────────┼──────────────────────────────┼───────────────────────────────┤
│    9841378 │             413966 │              9427412 │            3611061 │                     0.0 │                          0.0 │            0.6330736407035681 │
└────────────┴────────────────────┴──────────────────────┴────────────────────┴─────────────────────────┴──────────────────────────────┴───────────────────

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


This dataset has an unbalanced client history. Different clients have different `gsc_data_start` and `ga4_data_start` dates, so the same calendar period does not provide the same amount of historical information for every client.

Some early rows are GSC-only or have unavailable GA4 data. When `ga4_data_available` is FALSE, zero-filled GA4 values must not be interpreted as zero engagement.

The query table uses a fixed 90-day window and contains last-30 and previous-30-day information. Its time window can overlap with the daily performance data, so feature and outcome windows must be aligned before joining the tables.

The data can measure associations and support ranking or decision-support, but it cannot by itself prove that a content change caused a later performance change.

The final June 2026 month is treated as a sealed test window rather than being used to develop the outcome definition.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.